# 대화 기억 RAG 실습

지나간 플레이어 발언을 저장하고, 현재 질문과 관련된 발언을 다시 찾는 작은 예제입니다.
**CPU 런타임에서 위부터 차례로 실행**할 수 있습니다. 생성 모델이나 GPU는 필요하지 않습니다.
처음 실행할 때는 패키지와 임베딩 모델을 내려받으므로 인터넷 연결이 필요합니다.

| 단계 | 하는 일 |
|---|---|
| 준비 | 라이브러리와 CPU 임베딩 모델 준비 |
| 저장 | 세션과 NPC별로 가상의 플레이어 발언 저장 |
| 검색 | 질문에 가까운 발언과 출처 정보 확인 |
| 분리 | 다른 세션·다른 NPC의 기록이 섞이지 않는지 확인 |
| 초기화 | 기존 세션을 폐기하고 빈 세션으로 다시 시작 |

예제 문장은 모두 가상의 게임 대사입니다. 실제 대화나 개인정보를 포함하지 않습니다.
**플레이어가 말했다는 기록은 확인된 게임 사실과 다릅니다.**
검색 점수는 문장 사이의 유사도를 살펴보는 값이며, 내용의 진실 여부나 정답률을 뜻하지 않습니다.


## 0. 라이브러리 준비

Chroma는 메모리 안에서 벡터를 찾고, Sentence Transformers는 문장을 벡터로 바꿉니다.
런타임에 준비된 PyTorch를 사용합니다. 이 셀에서 PyTorch를 별도로 교체하지 않습니다.
설치 후 재시작 안내가 나오면 런타임을 재시작하고 위부터 다시 실행합니다.


In [ ]:
%pip install -q "chromadb>=1,<2" "sentence-transformers>=5,<6"


## 1. CPU 임베딩 모델 준비

한국어를 포함한 여러 언어를 처리하는 `intfloat/multilingual-e5-small`을 CPU에 불러옵니다.
모델 파일은 아래 revision에 고정합니다. 모델 캐시는 실행 환경에 남을 수 있지만,
이후에 저장할 대화 기억은 디스크에 기록하지 않습니다.

E5는 질문 앞에 `query:`, 저장할 문장 앞에 `passage:`를 붙여 사용합니다.
이 접두사는 다음 단계의 저장·검색 함수가 붙이므로 예제 문장에는 직접 넣지 않습니다.


In [ ]:
from sentence_transformers import SentenceTransformer

# 단계 1: 같은 모델 버전을 CPU에 준비합니다.
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"
EMBEDDING_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    revision=EMBEDDING_REVISION,
    device="cpu",
    trust_remote_code=False,
)
print("CPU 임베딩 모델 준비 완료")


## 2. 저장·검색 함수 준비

이 셀에는 기억 모듈의 전체 코드가 들어 있습니다. 별도 Python 파일을 받거나
공개 저장소에서 실행 코드를 가져오지 않으므로 이 노트북만 복사해 사용할 수 있습니다.
함수를 정의하는 셀이므로, 실행 후 화면에 결과가 없어도 다음 단계로 진행하면 됩니다.

| 함수 | 역할 |
|---|---|
| `create_memory_session` | 새 세션 ID 발급 |
| `remember_player_message` | 플레이어 발언과 해당 NPC의 턴 번호 저장 |
| `memory_count` | 해당 세션·NPC의 기록 개수 확인 |
| `search_memory` | 같은 세션·같은 NPC 범위에서 관련 기록 검색 |
| `discard_memory_session` | 세션의 기억 폐기 |

Chroma는 `EphemeralClient`로 실행하며 영구 저장 경로를 지정하지 않습니다.
세션을 폐기하거나 런타임을 종료하면 이 예제의 대화 기억은 다시 불러올 수 없습니다.
임베딩·검색 결과를 포함한 객체는 이 런타임의 메모리에서 다룹니다.


In [ ]:
"""세션별 이전 발언과 확인된 공개 사건을 검색하는 임시 기억 저장소."""
from threading import RLock
from uuid import uuid4

import chromadb


# 파일을 만들지 않는 임시 DB입니다. 프로세스가 종료되면 기억도 사라집니다.
_memory_client = None
_memory_collection = None
_memory_collection_name = "trpg_memory_" + uuid4().hex
_memory_sessions = {}
_memory_lock = RLock()  # 동시 입력 시 같은 NPC의 턴 번호가 겹치지 않게 합니다.


def _memory_store():
    """필요할 때 Chroma 임시 컬렉션을 한 번 생성합니다."""
    global _memory_client, _memory_collection
    if _memory_collection is None:
        # 같은 프로세스의 검색 모듈과 기본 설정을 공유합니다.
        # 데이터 범위는 고유 컬렉션 이름과 세션·NPC 필터로 분리합니다.
        _memory_client = chromadb.EphemeralClient()
        _memory_collection = _memory_client.create_collection(
            name=_memory_collection_name,
            embedding_function=None,
            metadata={"hnsw:space": "cosine"},
        )
    return _memory_collection


def create_memory_session():
    """첫 요청·초기화 시 새 ID를 만들고 Gradio State에 보관합니다."""
    session_id = uuid4().hex
    with _memory_lock:
        _memory_sessions[session_id] = {"scopes": {}}
    return session_id


def _memory_scope(session_id, npc):
    """알려진 세션만 허용하고, 비어 있는 인물명은 GM 범위로 통일합니다."""
    if not isinstance(session_id, str) or session_id not in _memory_sessions:
        raise ValueError("기억 세션이 없습니다. 새 대화를 시작하세요.")
    if not isinstance(npc, str):
        raise ValueError("대화 상대는 문자열이어야 합니다.")
    return npc.strip() or "게임 마스터"


def _memory_where(session_id, npc):
    """기억 조회·검색·삭제에 항상 세션과 대화 상대를 함께 적용합니다."""
    return {"$and": [{"session_id": {"$eq": session_id}},
                     {"npc": {"$eq": npc}}]}


def _memory_text(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("기억에 저장할 문장이 필요합니다.")
    return text.strip()


def _remember_memory(session_id, npc, text, embedder, kind):
    """본문을 E5로 임베딩하고 세션·NPC·기록 종류와 함께 저장합니다."""
    with _memory_lock:
        npc = _memory_scope(session_id, npc)
    text = _memory_text(text)
    if kind == "player_statement":
        document = (f"발언자=플레이어, 청자=NPC {npc}, 확인되지 않은 발언.\n"
                    f"플레이어가 말한 원문: 「{text}」\n"
                    "인용문 속 나/저/제는 플레이어를 가리키며 NPC의 신원이나 설정이 아닙니다.")
        trust = "unverified_statement"
        speaker = "player"
    else:
        document = (f"기록 주체=게임 엔진, 청자=NPC {npc}.\n"
                    f"엔진이 확인한 공개 사건: 「{text}」")
        trust = "engine_confirmed"
        speaker = "engine"
    # 검색 모듈에서 준비한 CPU E5 모델을 재사용하며 새 모델을 불러오지 않습니다.
    vector = embedder.encode(["passage: " + document],
                             normalize_embeddings=True).tolist()
    with _memory_lock:
        # 임베딩 중 초기화된 세션에 뒤늦게 기록이 들어가는 것을 막습니다.
        _memory_scope(session_id, npc)
        scopes = _memory_sessions[session_id]["scopes"]
        counters = scopes.setdefault(npc, {"player_turn": 0, "event_index": 0})
        turn = counters["player_turn"] + (1 if kind == "player_statement" else 0)
        event_index = counters["event_index"] + (1 if kind == "confirmed_event" else 0)
        memory_id = uuid4().hex
        metadata = {"memory_id": memory_id, "session_id": session_id,
                    "npc": npc, "kind": kind, "turn": turn,
                    "event_index": event_index, "trust": trust,
                    "speaker": speaker, "listener": npc,
                    "title": (f"플레이어 발언 {turn} / 청자 {npc}" if kind == "player_statement"
                              else f"엔진 공개 사건 {event_index} / 청자 {npc}")}
        _memory_store().add(ids=[memory_id], documents=[document],
                            metadatas=[metadata], embeddings=vector)
        counters["player_turn"] = turn
        counters["event_index"] = event_index
    return turn


def remember_player_message(session_id, npc, text, embedder):
    """사용자 발언을 저장하고 해당 NPC에게 말한 발언 번호를 반환합니다.

    행동 선언과 사용자 주장은 확정 사건이 아닙니다. 원문을 발언으로만 저장합니다.
    생성된 AI 답변이나 LLM 요약은 이 함수에 넣지 않습니다.
    """
    return _remember_memory(session_id, npc, text, embedder, "player_statement")


def remember_confirmed_event(session_id, npc, text, embedder):
    """신뢰할 수 있는 엔진의 공개 사건만 저장하고 현재 사용자 턴을 반환합니다.

    사용자 주장이나 생성문을 넣어 확정 사건으로 바꾸는 함수가 아닙니다.
    사건 번호는 따로 관리하므로 엔진 사건을 추가해도 사용자 턴은 증가하지 않습니다.
    """
    return _remember_memory(session_id, npc, text, embedder, "confirmed_event")


def memory_count(session_id, npc):
    """해당 세션·NPC의 발언과 확인된 사건 수를 반환합니다."""
    with _memory_lock:
        npc = _memory_scope(session_id, npc)
        if npc not in _memory_sessions[session_id]["scopes"]:
            return 0
        return len(_memory_store().get(
            where=_memory_where(session_id, npc), include=[])["ids"])


def search_memory(session_id, npc, question, embedder, k=3, exclude_recent_turns=6):
    """같은 세션·NPC의 기억을 검색해 payload용 memory_results 목록을 반환합니다.

    최근 여섯 사용자 발언은 최근 12메시지와 겹치므로 기본 검색에서 제외합니다.
    확인된 엔진 사건은 별도로 검색합니다. score는 코사인 유사도이며 정답 확률이 아닙니다.
    """
    if type(k) is not int or k < 0:
        raise ValueError("검색 개수 k는 0 이상의 정수여야 합니다.")
    if type(exclude_recent_turns) is not int or exclude_recent_turns < 0:
        raise ValueError("최근 발언 제외 수는 0 이상의 정수여야 합니다.")
    question = _memory_text(question)
    with _memory_lock:
        npc = _memory_scope(session_id, npc)
        scope = _memory_sessions[session_id]["scopes"].get(npc)
        if scope is None or k == 0:
            return []
        cutoff = scope["player_turn"] - exclude_recent_turns
        where = {"$and": [
            _memory_where(session_id, npc),
            {"$or": [
                {"kind": {"$eq": "confirmed_event"}},
                {"$and": [{"kind": {"$eq": "player_statement"}},
                          {"turn": {"$lte": cutoff}}]},
            ]},
        ]}
        collection = _memory_store()
        candidates = collection.get(where=where, include=[])["ids"]
        if not candidates:
            return []
    vector = embedder.encode(["query: " + question],
                             normalize_embeddings=True).tolist()
    with _memory_lock:
        _memory_scope(session_id, npc)
        result = collection.query(
            query_embeddings=vector, n_results=min(k, len(candidates)),
            where=where, include=["documents", "metadatas", "distances"])
    memories = []
    for text, metadata, distance in zip(result["documents"][0],
                                         result["metadatas"][0], result["distances"][0]):
        # 세션 ID는 저장소 필터에만 사용하며 모델에게 전달하지 않습니다.
        metadata = {key: value for key, value in metadata.items() if key != "session_id"}
        memories.append({"source": "memory", "text": text,
                         "score": round(1.0 - float(distance), 6),
                         "metadata": metadata})
    return memories


def discard_memory_session(session_id):
    """세션에 속한 모든 NPC 기억을 지우고 ID를 폐기합니다."""
    with _memory_lock:
        _memory_scope(session_id, "")
        for npc in _memory_sessions[session_id]["scopes"]:
            _memory_store().delete(where=_memory_where(session_id, npc))
        del _memory_sessions[session_id]


## 3. 가상의 플레이어 발언 저장

세션은 한 번의 플레이 기록을 나누는 경계이고, NPC는 그 안의 대화 상대를 나누는 경계입니다.
여기서는 이름 대신 `기록원`이라는 역할 표시를 사용합니다.
저장할 때마다 이 NPC에 대한 플레이어 턴이 하나씩 늘어납니다.

`kind=player_statement`는 **플레이어가 이렇게 말했다**는 뜻입니다.
예를 들어 열쇠를 넣었다는 발언만으로 실제 인벤토리가 바뀌지는 않습니다.
AI가 만든 답변은 이 함수에 전달하지 않으며 장기 기억으로 저장하지 않습니다.


In [ ]:
# 단계 2: 새 세션에서 가상의 플레이어 발언만 저장합니다.
session_a = create_memory_session()
npc_label = "기록원"
player_utterances = [
    "청동 열쇠를 낡은 지도 주머니에 넣어 두었어요.",
    "작은 돌다리를 건너기 전에 붉은 천 조각을 챙겼어요.",
    "서쪽 복도에서 바람 소리를 들었다고 생각해요.",
    "등불 두 개 중 하나는 아직 기름이 남아 있는 것 같아요.",
    "손으로 그린 지도에는 둥근 방을 표시해 두었어요.",
    "다음 길목에서 푸른 리본을 표식으로 묶어 둘 생각이에요.",
    "빈 상자에 은색 단추가 있었다고 말했어요.",
    "둥근 방으로 돌아가서 낡은 주머니부터 찾아볼게요.",
]
for utterance in player_utterances:
    turn = remember_player_message(session_a, npc_label, utterance, embedder)
    print(f"플레이어 턴 {turn}: {utterance}")

assert memory_count(session_a, npc_label) == len(player_utterances)
print("저장된 기록 수:", memory_count(session_a, npc_label))


## 4. 관련 발언 찾기

이번에는 저장한 기록 전체를 살펴보려고 `exclude_recent_turns=0`으로 설정합니다.
질문을 벡터로 바꾼 뒤, 같은 세션·같은 NPC의 기록 안에서 가까운 후보를 찾습니다.

결과에는 본문과 함께 `kind`, `npc`, `turn`, `score`를 표시합니다.
`score`가 높다는 이유만으로 내용을 확정 사실로 바꾸지 않습니다.
후보를 답변에 사용할 때도 플레이어 발언인지, 엔진이 확정한 이벤트인지 구분해야 합니다.


In [ ]:
def show_memory_rows(rows):
    """출처 종류와 검색 점수를 본문과 함께 읽기 쉽게 표시합니다."""
    if not rows:
        print("검색된 기록 없음")
        return
    for index, row in enumerate(rows, start=1):
        metadata = row["metadata"]
        print(
            f"[M{index}] kind={metadata['kind']} | npc={metadata['npc']} | "
            f"turn={metadata['turn']} | score={float(row['score']):.4f}"
        )
        print(row["text"])
        print()

# 단계 3: 최근 턴도 포함해 관련 발언 세 개를 찾아봅니다.
question = "청동 열쇠를 어디에 넣어 두었다고 말했나요?"
memory_rows = search_memory(
    session_a,
    npc_label,
    question,
    embedder,
    k=3,
    exclude_recent_turns=0,
)
show_memory_rows(memory_rows)


## 5. 최근 대화와 오래된 기억 구분

출력 프로토타입은 최근 **12메시지**를 바로 대화 문맥에 넣습니다.
기억 검색은 그보다 오래된 플레이어 발언을 보충하는 용도로 사용할 수 있습니다.
기본값인 `exclude_recent_turns=6`은 해당 NPC의 최근 **6개 플레이어 턴**을 제외합니다.
플레이어와 AI가 한 번씩 말한 왕복 6회는 12메시지에 대응하지만,
메시지 수와 플레이어 턴 수는 서로 다른 기준입니다.

아래 예제에서는 플레이어 발언 8개 중 오래된 1·2턴만 검색 대상이 됩니다.
AI 답변을 저장하지 않아도, 최근 대화는 출력 모듈의 별도 이력으로 전달할 수 있습니다.
엔진에서 확정한 이벤트를 저장하는 경로는 플레이어 발언과 구분하며, 이 실습에서는 사용하지 않습니다.


In [ ]:
# 단계 4: 기본 제외 범위를 적용해 오래된 플레이어 발언만 찾습니다.
older_rows = search_memory(
    session_a,
    npc_label,
    question,
    embedder,
    k=3,
    exclude_recent_turns=6,
)
assert all(row["metadata"]["turn"] <= 2 for row in older_rows)
show_memory_rows(older_rows)


## 6. 다른 세션·다른 NPC의 기록 분리

같은 질문이라도 세션이나 NPC가 다르면 해당 범위만 검색합니다.
먼저 비어 있는 두 범위에서 결과가 없는지 확인하고,
서로 다른 예제 문장을 저장한 뒤 원래 범위에 섞이지 않는지 살펴봅니다.
아래 점검은 이 예제에서의 분리 동작을 확인하는 것이며 검색 정확도를 평가하는 시험은 아닙니다.


In [ ]:
# 단계 5: 세션 경계와 NPC 경계를 각각 확인합니다.
session_b = create_memory_session()
other_npc = "문지기"
empty_session_rows = search_memory(
    session_b, npc_label, question, embedder, exclude_recent_turns=0
)
empty_npc_rows = search_memory(
    session_a, other_npc, question, embedder, exclude_recent_turns=0
)
assert empty_session_rows == []
assert empty_npc_rows == []
print("다른 세션의 빈 범위:", empty_session_rows)
print("다른 NPC의 빈 범위:", empty_npc_rows)

other_session_text = "하얀 상자에 구리 종을 넣어 두었어요."
other_npc_text = "바닥의 파란 원 위에 돌 조각을 올려두었어요."
remember_player_message(session_b, npc_label, other_session_text, embedder)
remember_player_message(session_a, other_npc, other_npc_text, embedder)

original_rows = search_memory(
    session_a, npc_label, question, embedder, k=20, exclude_recent_turns=0
)
assert len(original_rows) == len(player_utterances)
assert all(row["metadata"]["npc"] == npc_label for row in original_rows)
assert all(
    other_session_text not in row["text"] and other_npc_text not in row["text"]
    for row in original_rows
)
print("원래 범위의 기록 수:", memory_count(session_a, npc_label))
print("다른 세션의 기록 수:", memory_count(session_b, npc_label))
print("다른 NPC의 기록 수:", memory_count(session_a, other_npc))
print("이 예제의 세 범위가 각각 분리되어 있습니다.")


## 7. 세션을 폐기하고 다시 시작

`discard_memory_session`은 해당 세션의 모든 NPC 기록을 폐기합니다.
폐기한 세션 ID는 다시 사용하지 않고 새 세션을 만듭니다.
새 세션에서는 앞에서 저장한 열쇠 발언도 검색되지 않습니다.

대화 기억을 파일로 저장하거나 세션 사이에서 자동 복원하는 기능은 없습니다.
이 실습은 저장·검색·분리·초기화 과정을 설명하며, 검색 품질이나 응답의 사실성을 보장하지 않습니다.


In [ ]:
# 단계 6: 기존 세션을 폐기한 뒤 빈 세션을 새로 만듭니다.
old_session = session_a
discard_memory_session(old_session)
try:
    memory_count(old_session, npc_label)
except ValueError:
    print("폐기한 세션은 더 이상 사용할 수 없습니다.")
else:
    raise AssertionError("폐기한 세션의 사용이 차단되어야 합니다.")

session_a = create_memory_session()
assert session_a != old_session
assert memory_count(session_a, npc_label) == 0
reset_rows = search_memory(
    session_a, npc_label, question, embedder, exclude_recent_turns=0
)
assert reset_rows == []
print("새 세션의 기록 수:", memory_count(session_a, npc_label))
show_memory_rows(reset_rows)

# 분리 확인에 사용한 다른 예제 세션도 정리합니다.
discard_memory_session(session_b)


## 이어서 실행하기

- [기억 RAG를 Colab에서 열기](https://colab.research.google.com/github/TRPG-masterbot/data/blob/main/BAE/Memory_RAG.ipynb)
- [출력 프로토타입을 Colab에서 열기](https://colab.research.google.com/github/TRPG-masterbot/data/blob/main/BAE/Output_Code.ipynb)

기억 실습은 CPU로 실행합니다. 출력 프로토타입의 생성 모델은 해당 노트북에 안내된 GPU 환경을 사용합니다.
